# Day 015 | Mini Project: USCS Soil Classification Tool

## 100 Days of ML for Geotechnical Engineering
**Phase 1 Capstone** | **Author:** Ripon Chandra Malo | **Date:** 2026-02-28

---

## Phase 1 Capstone Project

Over the past 14 days we learned: variables, conditionals, loops, strings, lists, dicts, functions, file I/O, error handling, OOP, modules, decorators, and regex. Today we combine **all of them** into one complete tool that every geotechnical engineer uses: the **Unified Soil Classification System (USCS)** classifier.

### What We Will Build

| Component | Phase 1 Concepts Used |
|-----------|----------------------|
| `USCSSoilClassifier` class | OOP (Day 010) |
| Classification logic | Conditionals (Day 003), Functions (Day 007) |
| Input validation | Error handling (Day 009), `raise` |
| Read CSV test data | File I/O (Day 008), strings (Day 005) |
| Process all samples | Loops (Day 004), lists & dicts (Day 006) |
| Logging decorator | Decorators (Day 012) |
| Parse sample IDs | Regex (Day 014) |
| ASCII Casagrande chart | Strings, loops, conditionals |
| Classification report | Everything together |

### The USCS Standard (ASTM D2487)

The Unified Soil Classification System is the **standard** system for classifying soils in geotechnical engineering. Every site investigation report, every foundation design, every slope stability analysis references USCS symbols like **GW**, **CL**, **SM**.

The classification decision tree:

```
                         SOIL SAMPLE
                             |
              Is >50% retained on #200 sieve?
              /                            \
           YES                              NO
     COARSE-GRAINED                   FINE-GRAINED
           |                                |
  >50% of coarse on #4?          Liquid Limit < 50?
     /         \                    /            \
   YES          NO               YES              NO
  GRAVEL       SAND           LOW PLASTICITY   HIGH PLASTICITY
   G___        S___            CL/ML/OL         CH/MH/OH
     |           |                  |
  Fines%?     Fines%?        Above or below
  <5  5-12  >12              A-line on
  W/P  W/P   M/C            Casagrande chart?
```

---

## Part 1: USCS Symbol Reference

Before writing code, let's define what each USCS symbol means:

| Symbol | Full Name | Key Criteria |
|--------|-----------|-------------|
| **GW** | Well-graded gravel | Gravel, fines <5%, Cu>=4 and 1<=Cc<=3 |
| **GP** | Poorly-graded gravel | Gravel, fines <5%, fails Cu/Cc |
| **GM** | Silty gravel | Gravel, fines >12%, below A-line |
| **GC** | Clayey gravel | Gravel, fines >12%, above A-line |
| **GW-GM / GW-GC** | Well-graded gravel with silt/clay | Gravel, fines 5-12%, meets Cu/Cc |
| **GP-GM / GP-GC** | Poorly-graded gravel with silt/clay | Gravel, fines 5-12%, fails Cu/Cc |
| **SW** | Well-graded sand | Sand, fines <5%, Cu>=6 and 1<=Cc<=3 |
| **SP** | Poorly-graded sand | Sand, fines <5%, fails Cu/Cc |
| **SM** | Silty sand | Sand, fines >12%, below A-line |
| **SC** | Clayey sand | Sand, fines >12%, above A-line |
| **CL** | Lean clay | LL<50, above A-line, PI>7 |
| **ML** | Silt | LL<50, below A-line or PI<4 |
| **OL** | Organic silt/clay (low LL) | LL<50, organic |
| **CH** | Fat clay | LL>=50, above A-line |
| **MH** | Elastic silt | LL>=50, below A-line |
| **OH** | Organic clay/silt (high LL) | LL>=50, organic |
| **Pt** | Peat | Highly organic |

### The Casagrande Plasticity Chart

The A-line equation: **PI = 0.73 * (LL - 20)**

- **Above A-line** = Clay (C)
- **Below A-line** = Silt (M)

---

## Part 2: Helper Utilities — Decorator and Validation

We start by building reusable pieces: a logging decorator (Day 012) and input validation (Day 009).

In [ ]:
# ─────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────
import csv
import os
import re
from functools import wraps

print("Imports loaded: csv, os, re, functools.wraps")

In [ ]:
# ─────────────────────────────────────────────────────
# LOGGING DECORATOR (Day 012 concept)
# ─────────────────────────────────────────────────────

def log_classification(func):
    """Decorator that logs every classification call."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        # args[0] is self (the class instance)
        result = func(*args, **kwargs)
        sample_id = kwargs.get("sample_id", "unknown")
        print(f"  [LOG] classify() -> {result['symbol']:6s} | {result['name']}")
        return result
    return wrapper

print("Decorator defined: log_classification")
print("It wraps the classify method to print every result.")

In [ ]:
# ─────────────────────────────────────────────────────
# INPUT VALIDATION (Day 009 concept: raise ValueError)
# ─────────────────────────────────────────────────────

def validate_inputs(percent_gravel, percent_sand, percent_fines,
                    liquid_limit, plasticity_index, is_organic):
    """Validate soil data before classification.
    Raises ValueError with a clear message if anything is wrong.
    """
    # Check types
    for name, val in [("percent_gravel", percent_gravel),
                      ("percent_sand", percent_sand),
                      ("percent_fines", percent_fines),
                      ("liquid_limit", liquid_limit),
                      ("plasticity_index", plasticity_index)]:
        if not isinstance(val, (int, float)):
            raise TypeError(f"{name} must be a number, got {type(val).__name__}")
        if val < 0:
            raise ValueError(f"{name} cannot be negative (got {val})")

    # Percentages must sum to ~100
    total = percent_gravel + percent_sand + percent_fines
    if abs(total - 100.0) > 1.0:
        raise ValueError(
            f"Gravel + Sand + Fines must equal 100% "
            f"(got {percent_gravel} + {percent_sand} + {percent_fines} = {total})"
        )

    # Atterberg limits sanity checks
    if liquid_limit > 200:
        raise ValueError(f"Liquid limit {liquid_limit} unreasonably high (max ~200)")
    if plasticity_index > liquid_limit:
        raise ValueError(
            f"PI ({plasticity_index}) cannot exceed LL ({liquid_limit})"
        )

    return True


# Quick test
print("Validation tests:")
tests = [
    (60, 30, 10, 0, 0, False, "Valid coarse-grained"),
    (60, 30, 20, 0, 0, False, "Bad: sums to 110"),
    (-5, 80, 25, 0, 0, False, "Bad: negative gravel"),
    (5, 15, 80, 30, 50, False, "Bad: PI > LL"),
]

for g, s, f, ll, pi, org, desc in tests:
    try:
        validate_inputs(g, s, f, ll, pi, org)
        print(f"  OK  {desc}")
    except (ValueError, TypeError) as e:
        print(f"  ERR {desc}: {e}")

---

## Part 3: The USCSSoilClassifier Class

This is the core of our project. The class uses OOP (Day 010) to encapsulate all classification logic, with methods for classifying, describing, and visualizing results.

**Design decisions:**
- `classify()` returns a dict with `symbol`, `name`, and `group`
- `get_description()` returns a human-readable string
- `plot_on_casagrande_chart()` draws an ASCII chart (no external libraries)
- The decorator logs every classification automatically

In [ ]:
# ─────────────────────────────────────────────────────
# THE USCS SOIL CLASSIFIER CLASS (Day 010: OOP)
# ─────────────────────────────────────────────────────

class USCSSoilClassifier:
    """Classify soils according to the Unified Soil Classification System
    (ASTM D2487). Combines Phase 1 concepts: OOP, conditionals, functions,
    error handling, decorators, and string formatting.
    """

    # ── Class-level lookup table (dict — Day 006) ─────
    DESCRIPTIONS = {
        "GW":    "Well-graded gravel",
        "GP":    "Poorly-graded gravel",
        "GM":    "Silty gravel",
        "GC":    "Clayey gravel",
        "GW-GM": "Well-graded gravel with silt",
        "GW-GC": "Well-graded gravel with clay",
        "GP-GM": "Poorly-graded gravel with silt",
        "GP-GC": "Poorly-graded gravel with clay",
        "SW":    "Well-graded sand",
        "SP":    "Poorly-graded sand",
        "SM":    "Silty sand",
        "SC":    "Clayey sand",
        "SW-SM": "Well-graded sand with silt",
        "SW-SC": "Well-graded sand with clay",
        "SP-SM": "Poorly-graded sand with silt",
        "SP-SC": "Poorly-graded sand with clay",
        "CL":    "Lean clay",
        "ML":    "Silt",
        "OL":    "Organic clay or silt (low plasticity)",
        "CH":    "Fat clay",
        "MH":    "Elastic silt",
        "OH":    "Organic clay or silt (high plasticity)",
        "CL-ML": "Silty clay",
        "Pt":    "Peat and other highly organic soils",
    }

    def __init__(self):
        """Initialize the classifier with an empty results history."""
        self.history = []  # list of all classifications (Day 006: lists)

    # ── A-line check (Casagrande chart) ───────────────
    def _above_a_line(self, liquid_limit, plasticity_index):
        """Return True if the point plots above the A-line.
        A-line equation: PI = 0.73 * (LL - 20)
        """
        a_line_pi = 0.73 * (liquid_limit - 20)
        return plasticity_index > a_line_pi

    # ── Gradation check ───────────────────────────────
    def _is_well_graded(self, Cu, Cc, soil_type):
        """Check Cu and Cc criteria for well-graded classification.
        Gravel: Cu >= 4 and 1 <= Cc <= 3
        Sand:   Cu >= 6 and 1 <= Cc <= 3
        """
        cu_min = 4 if soil_type == "gravel" else 6
        return Cu >= cu_min and 1 <= Cc <= 3

    # ── Fine-grained classification ───────────────────
    def _classify_fine_grained(self, liquid_limit, plasticity_index, is_organic):
        """Classify fine-grained soil (>50% passing #200)."""
        if is_organic:
            return "OL" if liquid_limit < 50 else "OH"

        if liquid_limit < 50:
            # Low plasticity
            if self._above_a_line(liquid_limit, plasticity_index) and plasticity_index > 7:
                return "CL"
            elif not self._above_a_line(liquid_limit, plasticity_index) or plasticity_index < 4:
                return "ML"
            else:
                return "CL-ML"  # hatched zone on the chart
        else:
            # High plasticity
            if self._above_a_line(liquid_limit, plasticity_index):
                return "CH"
            else:
                return "MH"

    # ── Coarse-grained classification ─────────────────
    def _classify_coarse_grained(self, percent_gravel, percent_sand,
                                  percent_fines, liquid_limit,
                                  plasticity_index, Cu, Cc):
        """Classify coarse-grained soil (>50% retained on #200)."""
        coarse_total = percent_gravel + percent_sand

        # Gravel vs Sand: which dominates the coarse fraction?
        if percent_gravel >= percent_sand:
            prefix = "G"  # Gravel
            soil_type = "gravel"
        else:
            prefix = "S"  # Sand
            soil_type = "sand"

        well_graded = self._is_well_graded(Cu, Cc, soil_type)

        # Subdivision by fines content
        if percent_fines < 5:
            # Clean coarse-grained: W or P
            return f"{prefix}W" if well_graded else f"{prefix}P"

        elif percent_fines > 12:
            # Coarse-grained with fines: M or C
            if self._above_a_line(liquid_limit, plasticity_index) and plasticity_index > 7:
                return f"{prefix}C"  # clay fines
            else:
                return f"{prefix}M"  # silt fines

        else:
            # Dual symbol (5-12% fines)
            grad = "W" if well_graded else "P"
            if self._above_a_line(liquid_limit, plasticity_index) and plasticity_index > 7:
                fines_type = "C"
            else:
                fines_type = "M"
            return f"{prefix}{grad}-{prefix}{fines_type}"

    # ── Main classify method (with decorator) ─────────
    @log_classification
    def classify(self, percent_gravel, percent_sand, percent_fines,
                 liquid_limit, plasticity_index, is_organic=False,
                 Cu=0, Cc=0, sample_id=""):
        """Classify a soil sample according to USCS.

        Parameters
        ----------
        percent_gravel : float  -- % retained on #4 sieve
        percent_sand   : float  -- % passing #4, retained on #200
        percent_fines  : float  -- % passing #200 sieve
        liquid_limit   : float  -- Atterberg liquid limit
        plasticity_index : float -- Atterberg PI
        is_organic     : bool   -- True if organic content detected
        Cu             : float  -- Coefficient of uniformity (D60/D10)
        Cc             : float  -- Coefficient of curvature (D30^2 / D10*D60)
        sample_id      : str    -- Optional sample identifier

        Returns
        -------
        dict with 'symbol', 'name', and 'group'
        """
        # Step 1: Validate (Day 009)
        validate_inputs(percent_gravel, percent_sand, percent_fines,
                        liquid_limit, plasticity_index, is_organic)

        # Step 2: Coarse vs Fine (the fundamental split)
        if percent_fines <= 50:
            # COARSE-GRAINED
            symbol = self._classify_coarse_grained(
                percent_gravel, percent_sand, percent_fines,
                liquid_limit, plasticity_index, Cu, Cc
            )
            group = "Coarse-grained"
        else:
            # FINE-GRAINED
            symbol = self._classify_fine_grained(
                liquid_limit, plasticity_index, is_organic
            )
            group = "Fine-grained"

        # Step 3: Build result
        name = self.DESCRIPTIONS.get(symbol, "Unknown")
        result = {
            "sample_id": sample_id,
            "symbol": symbol,
            "name": name,
            "group": group,
        }

        # Step 4: Save to history
        self.history.append(result)

        return result

    # ── Description method ────────────────────────────
    def get_description(self, symbol):
        """Return the full USCS description for a given symbol."""
        return self.DESCRIPTIONS.get(symbol, f"Unknown symbol: {symbol}")

    # ── String representation (OOP) ───────────────────
    def __repr__(self):
        return f"USCSSoilClassifier(samples_classified={len(self.history)})"


print("USCSSoilClassifier class defined.")
print("Methods: classify(), get_description(), __repr__()")

---

## Part 4: Quick Test — Classify Known Soil Types

Before processing a full dataset, let's verify the classifier works on known soil types.

In [ ]:
# ─────────────────────────────────────────────────────
# QUICK TESTS — Known soil types
# ─────────────────────────────────────────────────────

clf = USCSSoilClassifier()

print("Test 1: Clean well-graded gravel (should be GW)")
r1 = clf.classify(65, 32, 3, 0, 0, Cu=10, Cc=2.0, sample_id="test-1")

print("\nTest 2: Fat clay (should be CH)")
r2 = clf.classify(2, 8, 90, 60, 35, sample_id="test-2")

print("\nTest 3: Silty sand (should be SM)")
r3 = clf.classify(5, 70, 25, 28, 5, sample_id="test-3")

print("\nTest 4: Organic soil, high LL (should be OH)")
r4 = clf.classify(1, 10, 89, 62, 35, is_organic=True, sample_id="test-4")

print(f"\n{clf}")
print(f"History has {len(clf.history)} entries.")

---

## Part 5: ASCII Casagrande Plasticity Chart

The Casagrande chart is the fundamental visual tool for fine-grained soil classification. We build an ASCII version using loops and string formatting (no external plotting libraries).

The **A-line** divides clays (above) from silts (below):
- A-line: PI = 0.73 * (LL - 20)

The **U-line** is the upper boundary (approximate):
- U-line: PI = 0.9 * (LL - 8)

In [ ]:
# ─────────────────────────────────────────────────────
# ASCII CASAGRANDE CHART (loops + strings + conditionals)
# ─────────────────────────────────────────────────────

def plot_casagrande_ascii(samples=None):
    """Draw an ASCII Casagrande plasticity chart.

    Parameters
    ----------
    samples : list of dict, optional
        Each dict needs 'liquid_limit', 'plasticity_index', 'symbol'.
        Points are plotted as numbers (1, 2, 3...).
    """
    # Chart dimensions
    width = 72       # characters wide (represents LL 0 to 100)
    height = 36      # characters tall (represents PI 0 to 60)
    ll_max = 100
    pi_max = 60

    # Create empty grid
    grid = [[" " for _ in range(width)] for _ in range(height)]

    # Helper: convert LL, PI to grid coordinates
    def to_col(ll):
        return int(ll / ll_max * (width - 1))

    def to_row(pi):
        return int((pi_max - pi) / pi_max * (height - 1))

    # Draw A-line: PI = 0.73 * (LL - 20)
    for ll in range(20, ll_max + 1):
        pi = 0.73 * (ll - 20)
        if 0 <= pi <= pi_max:
            c, r = to_col(ll), to_row(pi)
            if 0 <= r < height and 0 <= c < width:
                grid[r][c] = "/"

    # Draw U-line: PI = 0.9 * (LL - 8)
    for ll in range(8, ll_max + 1):
        pi = 0.9 * (ll - 8)
        if 0 <= pi <= pi_max:
            c, r = to_col(ll), to_row(pi)
            if 0 <= r < height and 0 <= c < width:
                grid[r][c] = ":"

    # Draw LL = 50 vertical line
    col_50 = to_col(50)
    for r in range(height):
        if grid[r][col_50] == " ":
            grid[r][col_50] = "|"

    # Place zone labels
    def place_label(ll, pi, text):
        c, r = to_col(ll), to_row(pi)
        for i, ch in enumerate(text):
            if 0 <= r < height and 0 <= c + i < width:
                grid[r][c + i] = ch

    place_label(30, 25, "CL")
    place_label(30, 5,  "ML")
    place_label(65, 40, "CH")
    place_label(65, 15, "MH")

    # Plot sample points (if provided)
    if samples:
        for i, s in enumerate(samples):
            ll = s.get("liquid_limit", 0)
            pi = s.get("plasticity_index", 0)
            if ll > 0 and pi > 0:  # only plot fine-grained
                c, r = to_col(ll), to_row(pi)
                if 0 <= r < height and 0 <= c < width:
                    grid[r][c] = str((i + 1) % 10)  # digit marker

    # Print chart
    print("    CASAGRANDE PLASTICITY CHART")
    print("    PI")
    for r in range(height):
        pi_val = pi_max - r * (pi_max / (height - 1))
        if r % 6 == 0:
            label = f"{pi_val:4.0f}|"
        else:
            label = "    |"
        print(label + "".join(grid[r]))
    print("    +" + "-" * width)
    print("     0         20        40    |   60        80       100")
    print("                              LL=50                    LL")
    print()
    print("    /  = A-line: PI = 0.73*(LL-20)")
    print("    :  = U-line: PI = 0.9*(LL-8)")

    if samples:
        print("\n    Sample points:")
        for i, s in enumerate(samples):
            ll = s.get("liquid_limit", 0)
            pi = s.get("plasticity_index", 0)
            sym = s.get("symbol", "?")
            if ll > 0 and pi > 0:
                print(f"      {(i+1) % 10} = {s.get('sample_id','?'):5s}  "
                      f"LL={ll:4.0f}, PI={pi:4.0f} -> {sym}")


# Draw the empty chart first
plot_casagrande_ascii()

---

## Part 6: Load Test Data from CSV

We have a file `soil_samples.csv` with 10 realistic soil samples. Let's read it using file I/O (Day 008) with error handling (Day 009) and regex (Day 014) for parsing sample IDs.

In [ ]:
# ─────────────────────────────────────────────────────
# LOAD CSV DATA (Day 008: file I/O, Day 009: error handling)
# ─────────────────────────────────────────────────────

def load_soil_samples(filepath):
    """Load soil samples from CSV.
    Returns a list of dicts with all values converted to proper types.
    """
    samples = []
    errors = []

    try:
        with open(filepath, "r") as f:
            reader = csv.DictReader(f)
            for i, row in enumerate(reader, start=1):
                try:
                    # Use regex (Day 014) to validate sample ID format
                    sid = row["sample_id"].strip()
                    if not re.match(r"^S-\d{2,3}$", sid):
                        raise ValueError(f"Bad sample_id format: '{sid}'")

                    sample = {
                        "sample_id":        sid,
                        "percent_gravel":   float(row["percent_gravel"]),
                        "percent_sand":     float(row["percent_sand"]),
                        "percent_fines":    float(row["percent_fines"]),
                        "liquid_limit":     float(row["liquid_limit"]),
                        "plasticity_index": float(row["plasticity_index"]),
                        "is_organic":       row["is_organic"].strip() == "True",
                        "Cu":               float(row["Cu"]),
                        "Cc":               float(row["Cc"]),
                    }
                    samples.append(sample)

                except (ValueError, KeyError) as e:
                    errors.append(f"Row {i}: {e}")

    except FileNotFoundError:
        print(f"ERROR: File not found: {filepath}")
        return [], [f"File not found: {filepath}"]

    return samples, errors


# Load our test data
samples, load_errors = load_soil_samples("soil_samples.csv")

print(f"Loaded {len(samples)} samples, {len(load_errors)} errors\n")

# Display what we loaded
print(f"{'ID':5s} {'Gravel':>7s} {'Sand':>6s} {'Fines':>6s} {'LL':>5s} {'PI':>5s} {'Org':>4s} {'Cu':>5s} {'Cc':>5s}")
print("-" * 55)
for s in samples:
    print(f"{s['sample_id']:5s} {s['percent_gravel']:6.1f}% {s['percent_sand']:5.1f}% "
          f"{s['percent_fines']:5.1f}% {s['liquid_limit']:5.1f} {s['plasticity_index']:5.1f} "
          f"{'Y' if s['is_organic'] else 'N':>4s} {s['Cu']:5.1f} {s['Cc']:5.1f}")

---

## Part 7: Classify All Samples

Now we loop through every sample (Day 004) and classify it. The decorator will log each result automatically.

In [ ]:
# ─────────────────────────────────────────────────────
# CLASSIFY ALL 10 SAMPLES (Day 004: loops)
# ─────────────────────────────────────────────────────

classifier = USCSSoilClassifier()
results = []
classify_errors = []

print("Classifying all samples...\n")

for s in samples:
    try:
        result = classifier.classify(
            percent_gravel=s["percent_gravel"],
            percent_sand=s["percent_sand"],
            percent_fines=s["percent_fines"],
            liquid_limit=s["liquid_limit"],
            plasticity_index=s["plasticity_index"],
            is_organic=s["is_organic"],
            Cu=s["Cu"],
            Cc=s["Cc"],
            sample_id=s["sample_id"],
        )
        # Merge input data with result for reporting
        result.update(s)
        results.append(result)

    except (ValueError, TypeError) as e:
        classify_errors.append(f"{s['sample_id']}: {e}")

print(f"\nClassified: {len(results)} | Errors: {len(classify_errors)}")
print(f"{classifier}")

---

## Part 8: Classification Report

Generate a formatted report — the kind you would attach to a geotechnical site investigation.

In [ ]:
# ─────────────────────────────────────────────────────
# CLASSIFICATION REPORT (strings + loops + formatting)
# ─────────────────────────────────────────────────────

def generate_report(results):
    """Generate a formatted USCS classification report."""
    lines = []
    lines.append("=" * 78)
    lines.append("             USCS SOIL CLASSIFICATION REPORT")
    lines.append("             ASTM D2487 -- Unified Soil Classification System")
    lines.append("=" * 78)
    lines.append("")

    # Summary table header
    hdr = (f"{'ID':5s} {'Gravel':>7s} {'Sand':>6s} {'Fines':>6s} "
           f"{'LL':>5s} {'PI':>5s} {'USCS':>7s} {'Description'}")
    lines.append(hdr)
    lines.append("-" * 78)

    for r in results:
        row = (f"{r['sample_id']:5s} {r['percent_gravel']:6.1f}% "
               f"{r['percent_sand']:5.1f}% {r['percent_fines']:5.1f}% "
               f"{r['liquid_limit']:5.1f} {r['plasticity_index']:5.1f} "
               f"{r['symbol']:>7s} {r['name']}")
        lines.append(row)

    lines.append("-" * 78)
    lines.append("")

    # Statistics summary
    coarse = [r for r in results if r["group"] == "Coarse-grained"]
    fine   = [r for r in results if r["group"] == "Fine-grained"]

    lines.append("SUMMARY")
    lines.append(f"  Total samples classified:  {len(results)}")
    lines.append(f"  Coarse-grained soils:      {len(coarse)}")
    lines.append(f"  Fine-grained soils:        {len(fine)}")
    lines.append("")

    # Count each symbol (dict — Day 006)
    symbol_counts = {}
    for r in results:
        sym = r["symbol"]
        symbol_counts[sym] = symbol_counts.get(sym, 0) + 1

    lines.append("SYMBOL DISTRIBUTION")
    for sym, count in sorted(symbol_counts.items()):
        bar = "#" * (count * 4)
        lines.append(f"  {sym:7s} {count:2d}  {bar}")

    lines.append("")
    lines.append("=" * 78)

    return "\n".join(lines)


report = generate_report(results)
print(report)

---

## Part 9: Plot Samples on the Casagrande Chart

Now we plot our fine-grained samples on the ASCII Casagrande chart to visually confirm the classification.

In [ ]:
# ─────────────────────────────────────────────────────
# PLOT FINE-GRAINED SAMPLES ON THE CASAGRANDE CHART
# ─────────────────────────────────────────────────────

# Only plot samples that have LL and PI > 0 (fine-grained relevant)
chart_samples = [
    r for r in results
    if r["liquid_limit"] > 0 and r["plasticity_index"] > 0
]

print(f"Plotting {len(chart_samples)} fine-grained samples on Casagrande chart:\n")
plot_casagrande_ascii(chart_samples)

---

## Part 10: Save Report to File

Write the classification report and results to output files (Day 008: file I/O).

In [ ]:
# ─────────────────────────────────────────────────────
# SAVE RESULTS (Day 008: File I/O)
# ─────────────────────────────────────────────────────

# Save the text report
with open("classification_report.txt", "w") as f:
    f.write(report)
print("Saved: classification_report.txt")

# Save results as CSV
output_fields = ["sample_id", "percent_gravel", "percent_sand",
                 "percent_fines", "liquid_limit", "plasticity_index",
                 "symbol", "name", "group"]

with open("classification_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=output_fields,
                            extrasaction="ignore")
    writer.writeheader()
    writer.writerows(results)
print("Saved: classification_results.csv")

# Verify by reading back
print("\nVerification -- reading back classification_results.csv:")
with open("classification_results.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(f"  {row['sample_id']} -> {row['symbol']:7s} {row['name']}")

---

## Part 11: Edge Cases and Error Handling

A robust tool handles bad input gracefully. Let's test edge cases to confirm our validation catches problems before they produce wrong results.

In [ ]:
# ─────────────────────────────────────────────────────
# EDGE CASES — What happens with bad data?
# ─────────────────────────────────────────────────────

edge_cases = [
    {"desc": "Percentages sum to 90",
     "args": (30, 50, 10, 0, 0)},
    {"desc": "Negative gravel",
     "args": (-5, 80, 25, 0, 0)},
    {"desc": "PI greater than LL",
     "args": (5, 15, 80, 30, 50)},
    {"desc": "String instead of number",
     "args": ("thirty", 50, 20, 0, 0)},
    {"desc": "LL = 250 (unreasonably high)",
     "args": (5, 15, 80, 250, 100)},
]

edge_clf = USCSSoilClassifier()

print("Edge Case Testing:")
print("=" * 60)
for case in edge_cases:
    try:
        r = edge_clf.classify(*case["args"], sample_id="edge")
        print(f"  OK   {case['desc']} -> {r['symbol']}")
    except (ValueError, TypeError) as e:
        print(f"  CAUGHT  {case['desc']}")
        print(f"          {type(e).__name__}: {e}")
    print()

print("All edge cases handled without crashing.")

---

## Part 12: Extending the Tool — Soil Description Generator

Real geotechnical reports don't just list a symbol. They provide a **full verbal description** of the soil. Let's add a method that generates one.

In [ ]:
# ─────────────────────────────────────────────────────
# SOIL DESCRIPTION GENERATOR (strings + conditionals)
# ─────────────────────────────────────────────────────

def generate_soil_description(result_dict):
    """Generate a verbal soil description from classification results.
    Example output: 'Silty SAND (SM) - 52% sand, 40% fines, 8% gravel;
                     LL=35, PI=12. Fine-grained fraction is low-plasticity silt.'
    """
    r = result_dict
    sym = r["symbol"]
    gravel = r["percent_gravel"]
    sand = r["percent_sand"]
    fines = r["percent_fines"]
    ll = r["liquid_limit"]
    pi = r["plasticity_index"]

    parts = []

    # Main soil name
    parts.append(f"{r['name']} ({sym})")

    # Grain size breakdown
    grain_parts = []
    if gravel > 0:
        grain_parts.append(f"{gravel:.0f}% gravel")
    if sand > 0:
        grain_parts.append(f"{sand:.0f}% sand")
    if fines > 0:
        grain_parts.append(f"{fines:.0f}% fines")
    parts.append(", ".join(grain_parts))

    # Atterberg limits (if applicable)
    if ll > 0 or pi > 0:
        parts.append(f"LL={ll:.0f}, PI={pi:.0f}")
        plasticity = "low" if ll < 50 else "high"
        parts.append(f"{plasticity}-plasticity fines")

    return "; ".join(parts)


# Generate descriptions for all classified samples
print("Soil Descriptions for Report:")
print("=" * 70)
for r in results:
    desc = generate_soil_description(r)
    print(f"\n  {r['sample_id']}: {desc}")
print()

---

## Key Takeaways

1. **USCS classification is a decision tree** — a series of if/elif/else checks on grain size percentages and Atterberg limits. This maps directly to Python conditionals.

2. **OOP organizes complex logic** — the `USCSSoilClassifier` class bundles classification methods, lookup tables, and result history into one reusable object.

3. **Input validation prevents silent errors** — checking that percentages sum to 100 and values are in range catches data entry mistakes before they produce wrong classifications.

4. **The skip-bad-keep-good pattern scales** — wrapping each sample's classification in try/except means one bad row does not kill the entire batch.

5. **Every Phase 1 concept appeared in this project** — variables stored soil properties, conditionals made decisions, loops processed batches, dicts mapped symbols to descriptions, file I/O loaded and saved data, decorators logged results, and regex validated IDs.

---

## Phase 1 Summary: Python Fundamentals (Days 001-015)

| Day | Topic | What We Learned |
|-----|-------|----------------|
| 001 | Variables & Data Types | int, float, str, bool for soil properties |
| 002 | Operators & Expressions | Arithmetic for bearing capacity, comparisons |
| 003 | Conditionals | if/elif/else for soil classification logic |
| 004 | Loops | for/while to process borehole data |
| 005 | Strings | Formatting reports, parsing field notes |
| 006 | Lists & Dictionaries | Storing layer data, lookup tables |
| 007 | Functions | Reusable SPT correlation functions |
| 008 | File I/O | Reading CSV borehole logs, writing reports |
| 009 | Error Handling | Surviving messy field data |
| 010 | OOP Basics | Borehole and SoilLayer classes |
| 011 | Modules & Packages | Organizing a geotech library |
| 012 | Decorators | Logging, timing, validation wrappers |
| 013 | List Comprehensions | Compact data transformations |
| 014 | Regular Expressions | Parsing borehole IDs and field notes |
| **015** | **Mini Project** | **USCS classifier using ALL concepts** |

---

## How This Connects to ML

| Phase 1 Concept | ML Equivalent in Phase 2+ |
|----------------|---------------------------|
| USCS decision tree (if/elif) | Decision Tree classifier (sklearn) |
| Manual feature thresholds (fines%, LL, PI) | Feature engineering for ML models |
| Classification into symbols (GW, CL, ...) | Multi-class classification |
| Casagrande chart (A-line boundary) | Decision boundary visualization |
| Input validation | Data preprocessing and cleaning |
| CSV data loading | pandas DataFrames |
| Processing 10 samples in a loop | Batch prediction on thousands of samples |
| Classification report | Confusion matrix, classification report |

**The big picture:** Today we manually coded every classification rule. In Phase 2, we will teach a machine learning model to **learn** these rules from data. Same problem, different approach — and the ML version can discover patterns that textbook rules miss.

---

*Day 015 of 100 — Phase 1 Complete!*